
# Step-by-Step ChromaDB Tutorial (Production-Oriented)

This notebook provides a **step-by-step introduction to ChromaDB**, a popular open‑source vector database used in Retrieval Augmented Generation (RAG) systems.

The tutorial gradually moves from **basic concepts to advanced usage**, including:

- Creating and managing vector collections
- Storing embeddings and documents
- Performing semantic search
- Using metadata filters
- Custom embedding models
- Updating and deleting documents
- Building a simple RAG retrieval pipeline

Notebook generated on: **2026-03-12**



## Step 1 — Install ChromaDB

Create a fresh Python environment and install required dependencies.

Required packages:

- chromadb
- sentence-transformers
- pandas (optional for data manipulation)
- langchain (optional for integration with LLM pipelines)

Run the following commands in your terminal before running the notebook.


In [ ]:

# Run these in a terminal if not already installed:
# pip install chromadb
# pip install sentence-transformers
# pip install pandas
# pip install langchain



## Step 2 — Initialize ChromaDB

ChromaDB supports two main modes:

1. **In-memory database** (good for experiments)
2. **Persistent database** (recommended for real applications)

Below we initialize a simple in-memory client.


In [ ]:

import chromadb

client = chromadb.Client()

print("ChromaDB client initialized")



## Step 3 — Create a Collection

A **collection** in ChromaDB is similar to a table in a relational database.

Each collection stores:

- documents
- embeddings
- metadata
- ids

We create a collection for company policy documents.


In [ ]:

collection = client.create_collection(name="company_policies")

print("Collection created")



## Step 4 — Insert Documents

Let’s simulate a small **enterprise policy dataset**.

These documents could represent:

- HR policies
- security policies
- remote work rules


In [ ]:

documents = [
    "Employees are entitled to 20 days of annual leave per year.",
    "Travel reimbursement requires manager approval.",
    "Passwords must be changed every 90 days.",
    "Employees can work remotely two days per week.",
    "Maternity leave is available for 26 weeks."
]

ids = ["doc1", "doc2", "doc3", "doc4", "doc5"]

collection.add(
    documents=documents,
    ids=ids
)

print("Documents inserted into ChromaDB")



## Step 5 — Perform Semantic Search

One of the key advantages of vector databases is **semantic similarity search**.

Instead of matching keywords, the system retrieves documents based on **meaning**.


In [ ]:

results = collection.query(
    query_texts=["How many leave days do employees get?"],
    n_results=2
)

results



## Step 6 — Use Metadata

Metadata allows us to attach structured information to each document.

Example metadata:

- department
- document type
- year


In [ ]:

collection.add(
    documents=[
        "Employees get 20 days of annual leave",
        "Travel expenses require approval",
        "Password must change every 90 days"
    ],
    ids=["m1", "m2", "m3"],
    metadatas=[
        {"department": "HR"},
        {"department": "Finance"},
        {"department": "IT"}
    ]
)

print("Documents with metadata inserted")



## Step 7 — Query with Metadata Filtering

Metadata filtering helps restrict search results to **specific categories**.


In [ ]:

results = collection.query(
    query_texts=["leave policy"],
    where={"department": "IT"}
)

results



## Step 8 — Use Custom Embedding Models

Instead of default embeddings, we can use models from **Sentence Transformers**.


In [ ]:
#!pip3 install sentence-transformers

In [ ]:
import sys
sys.executable

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model loaded")


## Step 9 — Create Collection with Custom Embeddings

We define a function that converts text into embeddings.


In [ ]:

'''
# Older method creates errors
custom_collection = client.create_collection(
    name="policy_docs",
    embedding_function=embed
)
'''
'''
custom_collection = client.get_or_create_collection(
    name="policy_docs",
    embedding_function=embed
)

'''


In [ ]:
class MyEmbeddingFunction:
    def __init__(self, model):
        self.model = model

    def __call__(self, input):
        # Used for documents
        return self.model.encode(input).tolist()

    # important for queries
    def embed_query(self, input):
        # Used for queries
        return self.model.encode(input).tolist()

embedding_fn = MyEmbeddingFunction(model)

# client.delete_collection(name="policy_docs")
custom_collection = client.create_collection(
    name="policy_docs",
    embedding_function=embedding_fn
)


## Step 10 — Batch Insert Documents

For large datasets we typically insert documents in batches.


In [ ]:

documents_batch = [
    "Annual leave policy for employees.",
    "Expense reimbursement guidelines.",
    "Security and password management rules.",
    "Remote work guidelines."
]

ids_batch = [f"id{i}" for i in range(len(documents_batch))]

custom_collection.add(
    documents=documents_batch,
    ids=ids_batch
)

print("Batch documents inserted")



## Step 11 — Store Embeddings Directly

Sometimes embeddings are generated externally (for example using OpenAI or other APIs).


In [ ]:

embeddings = model.encode(documents_batch).tolist()

custom_collection.add(
    embeddings=embeddings,
    documents=documents_batch,
    ids=[f"emb{i}" for i in range(len(documents_batch))]
)

print("Documents with external embeddings stored")



## Step 12 — Update and Delete Documents

ChromaDB supports updating or deleting documents.


In [ ]:

# Update
custom_collection.update(
    ids=["id0"],
    documents=["Employees get 25 days of annual leave per year"]
)

# Delete
custom_collection.delete(ids=["id1"])

print("Update and delete operations completed")



## Step 13 — Inspect Collection

Useful commands to inspect collections.


In [ ]:

print("Document count:", custom_collection.count())

print("Peek documents:")
custom_collection.peek()



## Step 14 — Build a Simple Retrieval Pipeline

In a RAG system, retrieval works as:

1. User query
2. Embedding generation
3. Vector search
4. Return top documents


In [ ]:

query = "What is the leave policy?"

results = custom_collection.query(
    query_texts=[query],
    n_results=3
)

context = results["documents"][0]

context



## Step 15 — Persistent Storage

For production systems, the vector database must persist on disk.


In [ ]:

import chromadb

persistent_client = chromadb.PersistentClient(path="./chroma_db")

collection = persistent_client.get_or_create_collection("persistent_docs")

collection.add(
    documents=["Example persistent document"],
    ids=["p1"]
)

print("✅ Database persisted automatically")
